## **This notebook shows how to plot the economic results** 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px

In [ ]:
#Read data from excel file
path = "PATH_TO_DATA_FILE" # --> Insert path to Data.xlsx file
sheet_name = 'cost_tkm'  # Replace 'YourSheetName' with the actual name of the sheet
results = pd.read_excel(path+'Data.xlsx', sheet_name=sheet_name, index_col=0)
results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# Assuming 'results' is your DataFrame
# Exclude 'Total Maximum' and 'Minimum' rows, then filter for 'Methanol' and 'SNG' rows
results_filtered = results.drop(index=['Total Maximum', 'Minimum'], errors='ignore')
selected_data = results_filtered.loc[['Methanol', 'SNG']]

# Exclude 'Maximum', 'Minimum', and 'Total' columns
columns_to_exclude = ['Maximum', 'Minimum', 'Total']
selected_data = selected_data.drop(columns=columns_to_exclude)

# Define a list of colors for the stacked bars
colors = ["#0e2c5cff", '#39627bff', '#1cadb7ff', '#87f6f2ff','#22d5e2ff', '#589da0ff', '#4be165ff', '#38aa4cff', '#215a1eff' ]  # Customize as needed

# Plot stacked bar for 'Methanol' and 'SNG' with custom colors
selected_data.plot(kind='bar', stacked=True, figsize=(6.5, 4), legend=True, color=colors)

# Extract 'Total' column values for 'SNG' and 'Methanol'
total_values = results_filtered.loc[['Methanol', 'SNG'], 'Total']

# Plot 'Total' values as scatter points
plt.scatter(x=total_values.index, y=total_values, color='black', zorder=6, s=60, label='Total')

# Iterate over each scenario to plot the lines for maximum and minimum
for scenario in ['Methanol', 'SNG']:
    # Extract the scenario-specific data
    scenario_data = results.loc[scenario, columns_to_exclude[:-1]]  # Exclude 'Total' from columns_to_exclude for this operation
    
    # Find maximum and minimum values and their corresponding categories
    max_value = scenario_data.max()
    min_value = scenario_data.min()
    
    # Determine the positions for the scenario in the plot
    scenario_pos = total_values.index.get_loc(scenario)
    
    # Plot horizontal lines for maximum and minimum without adding them to the legend
    plt.hlines(y=max_value, xmin=scenario_pos - 0.1, xmax=scenario_pos + 0.09, colors='black', linestyles='solid', label='_no_legend_')
    plt.hlines(y=min_value, xmin=scenario_pos - 0.1, xmax=scenario_pos + 0.09, colors='black', linestyles='solid', label='_no_legend_')
    
    # Connect maximum and minimum with a line
    plt.plot([scenario_pos, scenario_pos], [min_value, max_value], color='black', linestyle='-', linewidth=1.5)

# Plot 'BAU' as a dashed horizontal line without adding it to the legend
if 'BAU' in results_filtered.index:
    bau_total = results_filtered.loc['BAU', 'Total']
    plt.axhline(y=bau_total, color='black', linestyle='--')
    # Annotate the 'BAU' line at the right side end
    plt.text(x=plt.xlim()[1], y=bau_total, s=' HFO ship', verticalalignment='center', horizontalalignment='left', color='black', fontsize=10)

# Annotate the fuel production and capture on board for SNG
sng_index = selected_data.index.get_loc('SNG')
capture_columns = ['CAPEX capture on-board', 'Ammonia', 'Monoethanolamine']
fuel_production_columns = [col for col in selected_data.columns if col not in capture_columns]
capture_sum = results_filtered.loc['SNG', capture_columns].sum()
fuel_production_sum = results_filtered.loc['SNG', fuel_production_columns].sum()
y_pos_capture = capture_sum
y_pos_fuel_production = capture_sum + (fuel_production_sum / 2)

## Customize the plot
plt.ylabel(r'Unitary cost [USD tkm$\mathbf{^{-1}}$]', fontsize=10, fontname="Arial", fontweight='bold')
# Customize x-axis
x_ticks = [0, 1]
x_labels = ['Methanol \n ship', 'Natural gas \n ship']
plt.xticks(ticks=x_ticks, labels=x_labels, rotation=0, fontsize=10, fontname="Arial")
plt.yticks(fontsize=9, fontname="Arial")

# Remove x-axis ticks
plt.tick_params(axis='x', length=0)

# Set y-axis tick label thickness
plt.tick_params(axis='y', width=2)

# Place the legend outside the plot on the right side without a border
plt.legend(loc='upper left', bbox_to_anchor=(0.99, 1), frameon=False, fontsize=10, prop={'family': 'Arial'})# Remove right and upper borders and increase thickness of left and bottom borders
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

# Use ScalarFormatter for y-axis
ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))  # Use mathematical text for scientific notation

# Optionally, you can force the y-axis to use 'offset' (the scientific notation) if it's not automatically applied
ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

plt.tight_layout()
path2 = r"PATH_TO_FIGURES" # --> Insert path to save the figure
plt.savefig(path2+'economic_assessment.png', dpi=600, bbox_inches='tight')

# Show plot
plt.show()
